# Testing differrent ideass for including peer comparison un financial profile report

In [20]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
import requests
from neo4j import GraphDatabase
import numpy as np

In [2]:
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
NEO_PASSWORD = os.environ.get("NEO_PASSWORD")
NEO_USERNAME = os.environ.get("NEO_USERNAME")
NEO_URL = os.environ.get("NEO_URL")
NEO_DATABASE = os.environ.get("NEO_DATABASE")

client = OpenAI()

In [3]:
NEO_URL

'neo4j://172.17.96.1:7687'

In [26]:
# start by selecting the company
name="Walt Disney Co"
cik="0001744489"
ticker="DIS"

In [5]:
# graph initialization
driver = GraphDatabase.driver(
    NEO_URL,
    auth=(NEO_USERNAME, NEO_PASSWORD),
    database=NEO_DATABASE
)

In [ ]:
def get_prompts(company,company_cik,year,sample_report="",quantitative_data="",peer_quantitative_data="",peer_company_name="",issuer_table="",owner_table="",questions="",subgraph_triples="",report_v1="",report_v2="",report_v3=""):
    
    BASE = f"""You are an expert in finance and you are writing a financial credit risk report for the company {company} (with CIK {company_cik}) and year {year}. 
    You will generate a section called key rating drivers, which consists on a list of the main reasons behind the assigned credit risk rating, usually including strengths and weaknesses.
    Try to focus mainly in negative impacts and risks and avoid giving to many positive kwy drivers.
    Follow the style and structure of rating commentaries, with a main title for the key driver and a detailed description of it. 
    Note that the report should focus in 3 types of factors:
        - F1-Financial Profile: Quantitative indicators of company's financial strength, profitability, financial structure and financial flexibility.
        - F2-Business Profile: Internal strategic and organizational characteristics, including competitive positioning, managerial decisions, ownership and subsidiary structure, and governance quality.
        - F3-Operating Environment: External macroeconomic, sectoral, regulatory and other external conditions shaping the firm risk context.
    """

    V_ALL = f"""Generate the financial report focusing exclusively on the most relevant factors from these reports: {report_v1}, {report_v2}, {report_v3}. 
        
        Feel free to combine and relate key drivers from different reports.
        Focus in giving details on recent data and events.  
        
        Use a beautiful readable format, organizing the drivers in sections and including a short introduction and a conclusion.     
        """
    
    PROMPTS = {
        "v0":BASE,    
        
        "v1":BASE + f"""You will now focus only in key rating drivers related to the Financial Profile (F1)
                    The information provided in the generated report should come from the data you can find in this table: 
                    {quantitative_data}                    
                    and the answers to this questions:    
                    {questions}
                    
                    For peer comparison, use the data available from {peer_company_name}, which is in the same sector as {company}:
                    {peer_quantitative_data}
                    """,
                    # falta el peer comparison
                    
        "v2":BASE + f"""You will now focus only in key rating drivers related to the Business Profile (F2)
                    The information provided in the generated report should come from the data you can find in this tables: 
                    The first table is the issuer transactions table:
                    {issuer_table}
                    The second table is the owner transactions table:
                    {owner_table}
                    Generate your response based on the answers to this questions: 
                    {questions}
                    """,
                    # falta añadir las empresas subsidiarias del grafo
                    
        "v3":BASE + f"""You will now focus only in key rating drivers related to the Operating Environment (F3)
        The information provided in the generated report should come exclusively from the data you can find in this Knowledge Subgraph: 
        {subgraph_triples} 
        and the answers to this questions: 
        {questions}""",
        
        "v_all":BASE +  V_ALL,
        
        "v_all_with_sample": BASE +  V_ALL + f"""This is an example of a list with 3 key rating drivers (one of each type): 
        {sample_report}""",
    }

    return PROMPTS


def get_fitch_metrics_timeseries(ticker_symbol, report_year=2026, n_years=3):
    ticker = yf.Ticker(ticker_symbol)

    bs = ticker.balance_sheet      # Balance Sheet
    cf = ticker.cashflow           # Cash Flow
    is_ = ticker.financials        # Income Statement
    
    # cogemos los datos de los tres años anteriores a report_year
    # ejemplo: para 2025 cogeríamos desde 2022, 2023 y 2024
    # para 2026 -> 2023, 2024, 2025

    last_report_year = report_year - n_years

    # get all the columns until the last report year (it may be in different positions depending on the company data)

    cols = [col for col in bs.columns if last_report_year <= col.year < report_year]
    rows = []

    for col in cols:
        # --- Raw metrics ---
        revenue = is_.loc["Total Revenue", col] if "Total Revenue" in is_.index else None
        ebitda = is_.loc["EBITDA", col] if "EBITDA" in is_.index else None
        interest_exp = is_.loc["Interest Expense", col] if "Interest Expense" in is_.index else None

        total_debt = bs.loc["Total Debt", col] if "Total Debt" in bs.index else None
        cash = bs.loc["Cash And Cash Equivalents", col] if "Cash And Cash Equivalents" in bs.index else None
        st_debt = bs.loc["Current Debt", col] if "Current Debt" in bs.index else None
        net_debt = bs.loc["Net Debt", col] if "Net Debt" in bs.index else None

        cfo = cf.loc["Operating Cash Flow", col] if "Operating Cash Flow" in cf.index else None
        capex = cf.loc["Capital Expenditure", col] if "Capital Expenditure" in cf.index else None      
        fcf = cf.loc["Free Cash Flow", col] if "Free Cash Flow" in cf.index else None 
        
        ffo = cfo + cf.loc["Change In Working Capital",col]
        
        interest_paid = abs(cf.loc["Interest Paid Supplemental Data",col])
        preferred_divs = abs(cf.loc["Cash Dividends Paid",col])

        # --- Derived Fitch ratios ---
        row = {
            "Period": col.strftime("%Y-%m-%d") if hasattr(col, "strftime") else str(col),
            
            "EBITDA": ebitda,
            "Cash": cash,
            "CapEx":capex,

            # Margins
            "EBITDA Margin": (ebitda / revenue) if (ebitda and revenue) else None,

            # Cash flow levels
            "FFO": ffo,
            "CFO": cfo,
            "FCF": fcf,

            # Coverage
            "FFO Interest Coverage": (ffo + interest_paid + preferred_divs) / (interest_paid + preferred_divs),

            # Leverage
            "FFO leverage": (total_debt / (ffo + interest_paid + preferred_divs)) if (total_debt and ffo) else None,
            "EBITDA leverage": (total_debt/ebitda),

            # Free cash flow ratio
            "Free Cash Flow Ratio": (fcf / total_debt) if (fcf and total_debt) else None,

            # Liquidity
            "Cash/ST Debt": (cash / st_debt) if (cash and st_debt) else None,
        }

        rows.append(row)

    df = pd.DataFrame(rows).set_index("Period")
    return df

def generate_report(company,company_cik,year,prompt_type="v0",self_correct=True,**kwargs):

    first_prompt = get_prompts(company,company_cik,year,**kwargs)[prompt_type]
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "user", "content": first_prompt}
        ]
    )

    report_first = response.choices[0].message.content
    
    if not self_correct:
        return report_first
    
    messages = [
        {"role": "user", "content": first_prompt},         
        {"role": "assistant", "content": report_first},    
        {"role": "user", "content": """Now please correct and improve the previous report ignoring the list of questions from the first petition. Preserve the list format, without adding extra titles or sections. 
                                        - Eliminate any hallucinations, inaccuracies, or irrelevant information, leaving only the key factors. 
                                        - Make sure all key risks are included, and add any additional observations or insights from the data that were missed in the first response. 
                                        - Remove any point that seems ambiguous or not insightful."""}
    ]

    response_corrected = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    report_self_corrected = response_corrected.choices[0].message.content

    return report_first, report_self_corrected

In [ ]:
# get all the companies that belong to the same sector as walt disney
# nos quedamos con 10 por ejemplo y de esas 10 buscamos la que tenga un margen más parecido (diferencia absoluta más pequeña) -> assets size

def get_peers_from_neo(cik,limit=10):
    """Extract <limit> companies from the same sector as the company with cik=<cik>"""
    query = f"""
    MATCH p=(n:Company{{cik:"{cik}"}})-[:BELONGS_TO_INDUSTRY_OF]->()<-[:BELONGS_TO_INDUSTRY_OF]-(m:Company) RETURN m.ticker, m.name LIMIT {limit};"""

    with driver.session() as session:
        result = session.run(query, cik=cik)
        return result.data()

def get_company_size(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)
    bs = ticker.balance_sheet
    # we use the total assets as an indicator of the companies size (used by financial expert)
    size = bs.loc["Total Assets"].iloc[0]
    print(ticker_symbol,size)
    return size
    
def get_peer_data(company_cik,company_ticker):
    # we first extract a sample of copmanies from the same sector
    candidates = get_peers_from_neo(company_cik)
    
    # get the asset size, the selected peer will be the company with the closest asset size
    min_diff = np.inf
    size_diff = 0
    peer_ticker = ""
    peer_name = ""
    
    company_size = get_company_size(company_ticker)
    
    for candidate in candidates:
        size_diff = abs(company_size-get_company_size(candidate["m.ticker"]))
        print(size_diff)
        if size_diff < min_diff:
            peer_ticker = candidate["m.ticker"]
            peer_name = candidate["m.name"]
    
    # we select the company with smallest size difference
    # print("selected company",peer_name,peer_ticker)
    return peer_name, peer_ticker
    
    

get_peer_data(cik,ticker)

DIS 196219000000.0
MTN 5777885000.0
190441115000.0
MSGS 1472974000.0
194746026000.0
PRSU 845008000.0
195373992000.0
GDEN 1079906000.0
195139094000.0
selected company GOLDEN ENTERTAINMENT, INC. GDEN


In [ ]:
# generate v1 report
# get the quantitative data

f1_questions = """
**Profitability / Margin**
1. How has EBITDA margin trended over the past years, and does it indicate sufficient profitability to support debt repayment?
2. How does the company's profitability compare with peers within the same sector?
**Cash Flow**
3. Are FFO, CFO, and FCF consistent, and do they demonstrate sufficient cash generation to cover operational and financial obligations?
4. Are there signs of deterioration in cash conversion efficiency (EBITDA → CFO → FCF)?
**Coverage**
5. Is the interest  coverage ratio adequate to withstand potential declines in earnings?
6. How would coverage ratios change under a 10-20% decrease in FFO or CFO?
**Leverage**
7. What is the level and trend of leverage (Debt/FFO or Debt/EBITDA), and is it sustainable relative to cash generation?
8. How does near-term refinancing risk appear when evaluating Cash/ST Debt and Free Cash Flow ratio?
**Liquidity**
9. Does the company have sufficient cash and short-term resources to cover immediate obligations?
10. Are there potential liquidity pressures in the near term under declining cash flow scenarios (using Cash/ST Debt and FCF)?
**Efficiency / Cash Quality**
11. Are there discrepancies between EBITDA, CFO, and FCF that could indicate issues in cash generation quality?
12. Are there divergences between EBITDA growth and FCF growth, and what might they reveal about CapEx or working capital management?
**Sector / Peer Analysis**
13. How do the company's key metrics (EBITDA margin, FFO, leverage, liquidity) compare with peers within the same sector?
**Scenario / Stress Test**
14. Under adverse scenarios (e.g., reduced cash flow), do coverage, leverage, and liquidity metrics remain adequate?"""

report = {
        "company_name":"Walt Disney Co",
        "company_cik":"0001744489",
        "company_ticker":"DIS",
        "year":"2025"
    }

quantitative_data = get_fitch_metrics_timeseries(report["company_ticker"], report_year=report["year"])

# get peer data for peer comparison
peer_name, peer_ticker = get_peer_data(report["company_cik"],report["company_ticker"])
peer_quantitative_data = get_fitch_metrics_timeseries(peer_ticker, report_year=report["year"])

report_v1, report_v1_corrected = generate_report(company=report["company_name"],
                            company_cik=report["company_cik"],
                            year=report["year"],
                            prompt_type="v1",
                            questions=f1_questions,
                            quantitative_data = quantitative_data.to_markdown(),
                            peer_quantitative_data = peer_quantitative_data.to_markdown(),
                            peer_copmany_name=peer_name)

print("V1")
print("---------------------------------------------------")
print(quantitative_data.to_markdown())
print("---------------------------------------------------")
print(report_v1)
print("V1 corrected")
print("---------------------------------------------------")
print(report_v1_corrected)